In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import parser
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [ ]:
df1 = pd.read_csv('/content/Iteration3_SA_QLD.csv')
df2 = pd.read_csv('/content/Iteration3_VIC.csv')


df3= pd.read_csv('/content/Iteration3_Species_otherstates.csv')

In [ ]:
# df1 = df1.drop_duplicates(subset=['species_scientific_name','observation_date','latitude','longitude'])
# df2 = df2.drop_duplicates(subset=['species_scientific_name','observation_date','latitude','longitude'])

In [ ]:
df = pd.concat([df1, df2, df3], ignore_index=True)

In [ ]:
df.head(5)

,species_common_name,species_scientific_name,observation_date,month,state,latitude,longitude,observation_count,plant_common_name,plant_scientific_name,optimal_planting_months,growth_duration_weeks,flowering_period,sunlight_requirement,water_requirement,suitable_regions,is_endangered
0,Ostrich,Struthio camelus,9/11/2022,November,Queensland,-21.65029,138.11203,8,Kangaroo Paw,Anigozanthos spp.,Aug-Dec,12.0,Aug-Dec,Full Sun,Low,"QLD, SA",NaN
1,Ostrich,Struthio camelus,8/02/2022,February,Queensland,-13.09540,153.27852,6,Kangaroo Paw,Anigozanthos spp.,Aug-Dec,12.0,Aug-Dec,Full Sun,Low,"QLD, SA",NaN
2,Ostrich,Struthio camelus,9/12/2022,December,Queensland,-14.78864,146.63214,2,Bottlebrush,Callistemon spp.,Sep-Nov,10.0,Sep-Nov,Full Sun,Moderate,"QLD, SA",NaN
3,Ostrich,Struthio camelus,6/09/2022,September,Queensland,-11.38336,148.32343,13,Kangaroo Paw,Anigozanthos spp.,Aug-Dec,12.0,Aug-Dec,Full Sun,Low,"QLD, SA",NaN
4,Ostrich,Struthio camelus,1/01/2023,January,Queensland,-15.16180,144.77058,9,Bottlebrush,Callistemon spp.,Sep-Nov,10.0,Sep-Nov,Full Sun,Moderate,"QLD, SA",NaN


In [ ]:
# Check for null values in 'observation_date' column
null_count = df['observation_date'].isnull().sum()

if null_count > 0:
  print(f"There are {null_count} null values in the 'observation_date' column.")
else:
  print("There are no null values in the 'observation_date' column.")


There are 4605 null values in the 'observation_date' column.


In [ ]:
#check for missing values and duplicates

# Check for missing values
print(df.isnull().sum())
# Check for duplicates
print(df.duplicated().sum())


species_common_name          0
species_scientific_name    780
observation_date             0
month                        0
state                        0
latitude                     0
longitude                    0
observation_count            0
plant_common_name            0
plant_scientific_name        0
optimal_planting_months      0
growth_duration_weeks        0
flowering_period             0
sunlight_requirement         0
water_requirement            0
suitable_regions             0
dtype: int64
0


In [ ]:
df['state'].unique()

array(['Queensland', 'South Australia', 'Victoria'], dtype=object)

In [ ]:
#df = df.drop_duplicates(subset=['species_scientific_name','observation_date','latitude','longitude'])


In [ ]:
df.shape

(13815, 16)

In [ ]:
# 1. Flexible date parsing
df['observation_date'] = pd.to_datetime(
    df['observation_date'],
    dayfirst=True,
    infer_datetime_format=True,
    errors='coerce'
)

# 2. Check for any failures
bad = df['observation_date'].isna().sum()
print(f"Unparsed dates: {bad}")

# (Optional) If any, inspect a few:
print(df.loc[df['observation_date'].isna(), 'observation_date'].head())

# 3. Now derive year/month
df['year']       = df['observation_date'].dt.year
df['month_full'] = df['observation_date'].dt.month_name()
df['month']      = df['month_full'].str.slice(0,3)



Unparsed dates: 4605
9210   NaT
9211   NaT
9212   NaT
9213   NaT
9214   NaT
Name: observation_date, dtype: datetime64[ns]


<ipython-input-7-7f7b4b779daa>:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['observation_date'] = pd.to_datetime(


In [ ]:
df.loc[[9210, 9211, 9212, 9213, 9214]]

,species_common_name,species_scientific_name,observation_date,month,state,latitude,longitude,observation_count,plant_common_name,plant_scientific_name,optimal_planting_months,growth_duration_weeks,flowering_period,sunlight_requirement,water_requirement,suitable_regions,year,month_full
9210,Ostrich,Struthio camelus,NaT,NaN,Victoria,-35.54394,147.75139,6,Bottlebrush,Callistemon spp.,Sep-Nov,10,Sep-Nov,Full Sun,Moderate,VIC,NaN,NaN
9211,Ostrich,Struthio camelus,NaT,NaN,Victoria,-38.17660,142.00771,13,Lilly Pilly,Syzygium spp.,Nov-Jan,16,Nov-Jan,Partial Sun,High,VIC,NaN,NaN
9212,Ostrich,Struthio camelus,NaT,NaN,Victoria,-35.95892,147.56197,4,Grevillea,Grevillea spp.,Jul-Oct,14,Jul-Oct,Full Sun,Low,VIC,NaN,NaN
9213,Ostrich,Struthio camelus,NaT,NaN,Victoria,-36.65431,149.62634,2,Wattle,Acacia spp.,Aug-Sep,8,Aug-Sep,Full Sun,Low,VIC,NaN,NaN
9214,Ostrich,Struthio camelus,NaT,NaN,Victoria,-36.77289,146.42699,8,Lilly Pilly,Syzygium spp.,Nov-Jan,16,Nov-Jan,Partial Sun,High,VIC,NaN,NaN


In [ ]:
bad_dates = df.loc[df['observation_date'].isna(), 'observation_date']
print(bad_dates.head(50).to_list())


[NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT, NaT]


In [ ]:
df_clean = df.dropna(subset=['observation_date'])
print(f"Remaining valid records: {len(df_clean)}")

Remaining valid records: 9210


In [ ]:
df_clean = df.dropna(subset=['observation_date']).copy()


In [ ]:
#Derive year/month fields
df_clean['year']       = df_clean['observation_date'].dt.year
df_clean['month_full'] = df_clean['observation_date'].dt.month_name()
df_clean['month']      = df_clean['month_full'].str.slice(0,3)




In [ ]:
#Infer state by latitude/longitude
conds = [
    df_clean['latitude'].astype(float).between(-29, -10) & df_clean['longitude'].astype(float).between(138, 154),
    df_clean['latitude'].astype(float).between(-38, -25) & df_clean['longitude'].astype(float).between(129, 141),
    df_clean['latitude'].astype(float).between(-39, -34) & df_clean['longitude'].astype(float).between(141, 150),
]
choices = ['Queensland','South Australia','Victoria']
df_clean['state'] = np.select(conds, choices, default=df_clean.get('state'))

In [ ]:
#Fix observation counts & ensure plant columns
df_clean['observation_count'] = pd.to_numeric(df_clean['observation_count'], errors='coerce').fillna(1).astype(int)

for col in ['plant_common_name','optimal_planting_months','growth_duration_weeks',
            'sunlight_requirement','water_requirement','suitable_regions']:
    if col not in df_clean.columns:
        df_clean[col] = pd.NA


In [ ]:
#Flag migratory species (species seen in >1 state)
visits = df_clean.groupby('species_scientific_name')['state'].nunique()
df_clean['is_migratory'] = df_clean['species_scientific_name'].isin(visits[visits>1].index)

In [ ]:
df_clean.to_csv('merged_birds_cleaned.csv', index=False)
print("Cleaned dataset saved as merged_birds_plants_cleaned.csv")

Cleaned dataset saved as merged_birds_plants_cleaned.csv


**Merging Datasets to get Endangered species**

In [11]:
vic_df = pd.read_csv('/content/Iteration3_VIC_with_endangered_purpose.csv')
sa_qld_df = pd.read_csv('/content/Iteration3_SA_QLD_with_endangered_purpose.csv')
other_states_df = pd.read_csv('/content/Iteration3_Species_otherstates_with_purpose.csv')


In [12]:
merged_df = pd.concat([vic_df, sa_qld_df, other_states_df], ignore_index=True)

In [17]:
grouped_df = merged_df.groupby(
    ['species_common_name', 'species_scientific_name', 'state', 'month', 'is_endangered'],
    as_index=False
)['observation_count'].sum().rename(columns={'observation_count': 'total_observation_count'})


In [7]:
#grouped_df = grouped_df.sort_values(['species_common_name', 'state', 'month'])

In [18]:
grouped_df.head(5)
grouped_df.describe()

,total_observation_count
count,12535.000000
mean,90.487116
std,208.666512
min,1.000000
25%,6.000000
50%,10.000000
75%,15.000000
max,904.000000


In [19]:
grouped_df.to_csv('Iteration3_Grouped_Summary.csv', index=False)

**Modelling**

**Bird Attraction Predictor: Overview**


*Goal: Predict the likelihood of observing a particular bird species in a region (state + lat/lon) for a given month.*

In [ ]:
import lightgbm as lgb
import pandas as pd

In [ ]:
# Load data
df = pd.read_csv('/content/merged_birds_cleaned.csv')

# Define features & target
features = ['state', 'latitude', 'longitude', 'month', 'plant_common_name',
            'sunlight_requirement', 'water_requirement']
target = 'species_common_name'

# Drop NaNs in target
df = df.dropna(subset=[target])

# Label Encoding for all categorical columns
encoders = {}
for col in ['state', 'month', 'plant_common_name', 'sunlight_requirement', 'water_requirement']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le



In [ ]:
# Target Encoding
target_le = LabelEncoder()
df[target] = target_le.fit_transform(df[target])

# Train-Test Split
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Train the model
clf = lgb.LGBMClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

LGBMClassifier(random_state=42)

In [ ]:
print(df_clean['species_common_name'].nunique())
print(df_clean['state'].value_counts())


921
state
Queensland         4936
South Australia    4274
Name: count, dtype: int64


In [ ]:
import gc
gc.collect()


16

In [ ]:
user_input = pd.DataFrame([{
    'state': encoders['state'].transform(['Queensland'])[0],
    'latitude': -37.81,
    'longitude': 144.96,
    'month': encoders['month'].transform(['Mar'])[0],
    'plant_common_name': encoders['plant_common_name'].transform(['Bottlebrush'])[0],
    'sunlight_requirement': encoders['sunlight_requirement'].transform(['Full Sun'])[0],
    'water_requirement': encoders['water_requirement'].transform(['Moderate'])[0]
}])

# Get probabilities for all species
proba = clf.predict_proba(user_input)[0]

# Map probabilities back to species names
species_prob = pd.DataFrame({
    'species_common_name': target_le.inverse_transform(range(len(proba))),
    'probability': proba
}).sort_values(by='probability', ascending=False)

NameError: name 'encoders' is not defined

In [ ]:
species_prob.head(5)


NameError: name 'species_prob' is not defined